# 02 — Rubric generation

Plan and, only when explicitly enabled, generate rubric trajectories with the frozen updater. Notebook 01 must have prepared the run. The default controls perform zero model calls.

In [ ]:
from pathlib import Path

import pandas as pd

from rubric_order.api import (
    build_rubrics,
    find_project_root,
    get_call_records,
    initial_rubric,
    inspect_protocol,
    inspect_rubric_trajectory,
    load_experiment_config,
    plan_rubric_generation,
    summarize_calls,
)

PROJECT_ROOT = find_project_root(Path.cwd())

In [ ]:
DATA_PATH = PROJECT_ROOT / "HelpSteer2"
CONFIG_PATH = PROJECT_ROOT / "configs" / "pilot.yaml"
RUN_ID = "pilot-001"
RUNTIME_OVERRIDES = {}

EXECUTE_MODEL_CALLS = False
MAX_CALLS = 0
SELECTED_METHODS = ["static"]
SELECTED_BLOCKS = [0, 1]
SELECTED_ORDERS = ["ascending", "descending", "random", "reverse_random"]
SELECTED_REPLICATES = [0]

In [ ]:
config = load_experiment_config(CONFIG_PATH, DATA_PATH, overrides=RUNTIME_OVERRIDES)
protocol = inspect_protocol(config, RUN_ID)
print(
    f"Prepared run {RUN_ID}: {len(protocol['blocks'])} blocks, "
    f"{protocol['heldout_response_count']} held-out responses"
)
initial = initial_rubric(config)
initial

In [ ]:
selection = {
    "methods": SELECTED_METHODS,
    "blocks": SELECTED_BLOCKS,
    "orders": SELECTED_ORDERS,
    "replicates": SELECTED_REPLICATES,
}
plan = plan_rubric_generation(config, RUN_ID, **selection)
display(pd.DataFrame(plan["jobs"]))
{
    "model": plan["model"],
    "planned_calls": plan["planned_calls"],
    "estimated_max_generated_tokens": plan["estimated_max_generated_tokens"],
    "execution_enabled": EXECUTE_MODEL_CALLS,
    "max_calls": MAX_CALLS,
}

The default selection is the intended canary: one sequential trajectory, one block, one random order, and ten updater calls. Set `EXECUTE_MODEL_CALLS = True` and `MAX_CALLS = 10` only after inspecting the plan.

In [ ]:
if EXECUTE_MODEL_CALLS:
    generation_result = build_rubrics(
        config, RUN_ID, execute_model_calls=True, max_calls=MAX_CALLS, **selection
    )
else:
    generation_result = {**plan, "executed": False, "actual_calls": 0}
    print("No files changed. Inspect the in-memory plan before enabling execution.")
{
    key: generation_result.get(key)
    for key in ["executed", "planned_calls", "actual_calls", "cache_hits", "dry_run_manifest"]
}

In [ ]:
trajectories = []
for method in SELECTED_METHODS:
    for block in SELECTED_BLOCKS:
        for order in SELECTED_ORDERS:
            for replicate in SELECTED_REPLICATES:
                trajectory = inspect_rubric_trajectory(
                    config, RUN_ID, method=method, block=block, order=order, replicate=replicate
                )
                if trajectory["steps"]:
                    trajectories.append(trajectory)
                    print(method, block, order, replicate)
                    display(pd.DataFrame(trajectory["steps"]))
                    display(trajectory["final_rubric"])
if not trajectories:
    print("No matching rubric trajectory is stored yet. The dry-run default is working.")

In [ ]:
rubric_call_summary = summarize_calls(config, RUN_ID, stage="build-rubrics", **selection)
rubric_call_summary

In [ ]:
# Set an SQLite call-record ID to inspect one raw response explicitly.
RAW_RECORD_ID = None
if RAW_RECORD_ID is not None:
    raw_records = get_call_records(
        config, RUN_ID, stage="build-rubrics", include_raw=True, **selection
    )
    selected_raw = next((row for row in raw_records if row["id"] == RAW_RECORD_ID), None)
    if selected_raw is None:
        raise ValueError(f"No selected call record has ID {RAW_RECORD_ID}")
    display(selected_raw)